<a href="https://colab.research.google.com/github/ParrvLuthra22/Drishti/blob/main/notebooks/phase4_videomae_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 4 — VideoMAE Fight Detection Fine-tuning
- Model: MCG-NJU/videomae-base
- Dataset: RWF-2000 (1600 train, 400 val)
- Device: T4 GPU (Colab)


In [1]:
# Setup
!pip install torch torchvision transformers datasets scikit-learn supervision ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 7.2 MB/s eta 0:00:00


In [3]:
# Clone repo
import sys, os

!git clone https://github.com/ParrvLuthra22/drishti.git
%cd drishti
sys.path.insert(0, '/content/drishti/src')
print("Repo ready")

Cloning into 'drishti'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 97 (delta 35), reused 82 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (97/97), 85.89 KiB | 2.60 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/drishti
Repo ready


In [5]:
# Upload RWF-2000.zip directly from your Mac
from google.colab import files
import zipfile, os

os.makedirs('data', exist_ok=True)
print("Select RWF-2000.zip when the picker opens...")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f"Extracting {zip_name}...")
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('data/')

# Handle folder name
if os.path.exists('data/RWF-2000'):
    os.rename('data/RWF-2000', 'data/rwf2000')

print("\nDone. Verify:")
!ls data/rwf2000/
!ls data/rwf2000/train/
!ls data/rwf2000/val/


Select RWF-2000.zip when the picker opens...


KeyboardInterrupt: 

In [6]:
# Verify GPU
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
!nvidia-smi


True
Tesla T4
Thu Oct  1 20:15:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------------

In [7]:
# Train
import sys
sys.path.insert(0, '/content/drishti/src')
from drishti.actions.trainer import VideoMAETrainer

trainer = VideoMAETrainer(
    data_root='data/rwf2000',
    output_dir='models/videomae_fight',
    device='cuda',
    batch_size=4,
    num_epochs=5
)
trainer.train()


KeyboardInterrupt: 

In [ ]:
# Push to HuggingFace Hub
# Store a write token in Colab Secrets (key icon in the left sidebar) as HF_TOKEN, and enable
# notebook access. Do not paste the token into this notebook: it gets committed to GitHub.
from google.colab import userdata
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path='models/videomae_fight',
    repo_id='ParrvLuthra22/drishti-fight-detector',
    repo_type='model',
    token=userdata.get('HF_TOKEN')
)
print("Pushed to HuggingFace Hub")


In [ ]:
# Quick eval
trainer.evaluate('models/videomae_fight/best_model.pt')
